In [ ]:
import cash
import numpy as np
from scipy import sparse
from scipy.sparse.linalg import spsolve
import time
import os

In [ ]:
%cash_on
%cash_debug

## 1. Domain Setup & Physical Parameters

We define the computational domain for a **2D lid-driven cavity** — a classic CFD benchmark.
The lid (top wall) moves at velocity $U = 1$ while all other walls are stationary.

The governing equations are the 2D incompressible Navier-Stokes equations:

$$\frac{\partial \mathbf{u}}{\partial t} + (\mathbf{u} \cdot \nabla)\mathbf{u} = -\frac{1}{\rho}\nabla p + \nu \nabla^2 \mathbf{u}$$

$$\nabla \cdot \mathbf{u} = 0$$

We use the Reynolds number $Re = UL/\nu$ to characterize the flow regime.

> **💡 Try this:** Change `Re` below and re-run. Only this cell and its dependents recompute — the mesh generation in the next cell is cached!

In [ ]:
# @cash:persist
# Physical parameters — change Re to explore different flow regimes
Re = 100          # Reynolds number (try 100, 400, 1000)
U_lid = 1.0       # Lid velocity [m/s]
L = 1.0           # Cavity length [m]
nu = U_lid * L / Re  # Kinematic viscosity
nu = U_lid * L / Re  # Kinematic viscosity
rho = 1.0         # Density [kg/m³]
print(f"Reynolds number: {Re}")
print(f"Kinematic viscosity: {nu:.6f} m²/s")

# Computational grid
N = 41            # Grid points in each direction (41x41)
dx = L / (N - 1)
dy = L / (N - 1)
dt = 0.001        # Time step [s]
n_steps = 50000     # Number of time steps
print(f"Grid: {N}x{N}, dx=dy={dx:.4f}, dt={dt}, steps={n_steps}")

# Create coordinate arrays
x = np.linspace(0, L, N)
y = np.linspace(0, L, N)
X, Y = np.meshgrid(x, y)
print(f"Domain: [{x[0]}, {x[-1]}] x [{y[0]}, {y[-1]}]")

## 2. Solver Functions

We define the core numerical operators. The `@pure` decorator tells Cash these functions have
no side effects, so it can skip mutation detection and safely cache any code that calls them.

The pressure Poisson equation is solved iteratively:
$$\nabla^2 p = \frac{\rho}{\Delta t} \left( \frac{\partial u}{\partial x} + \frac{\partial v}{\partial y} \right)$$

In [ ]:
@cash.pure
def build_laplacian_2d(n, dx, dy):
    """Build the 2D Laplacian operator as a sparse matrix.
    
    Uses 5-point stencil: ∇²f ≈ (f_{i+1,j} + f_{i-1,j} + f_{i,j+1} + f_{i,j-1} - 4f_{i,j}) / h²
    """
    n2 = n * n
    diags = np.zeros((5, n2))
    
    # Main diagonal
    diags[2, :] = -2.0 / dx**2 - 2.0 / dy**2
    # Off-diagonals (x-direction)
    diags[1, :] = 1.0 / dx**2  # i+1
    diags[3, :] = 1.0 / dx**2  # i-1
    # Off-diagonals (y-direction)
    diags[0, :] = 1.0 / dy**2  # j+1
    diags[4, :] = 1.0 / dy**2  # j-1
    
    offsets = [n, 1, 0, -1, -n]
    A = sparse.diags(diags, offsets, shape=(n2, n2), format='csc')
    return A

@cash.pure
def compute_divergence(u, v, dx, dy):
    """Compute velocity divergence field: ∂u/∂x + ∂v/∂y."""
    div = np.zeros_like(u)
    div[1:-1, 1:-1] = (
        (u[1:-1, 2:] - u[1:-1, :-2]) / (2 * dx) +
        (v[2:, 1:-1] - v[:-2, 1:-1]) / (2 * dy)
    )
    return div

@cash.pure
def apply_boundary_conditions(u, v, U_lid):
    """Apply no-slip walls + moving lid (top boundary)."""
    # No-slip on all walls
    u[0, :] = 0;  u[-1, :] = 0;  u[:, 0] = 0;  u[:, -1] = 0
    v[0, :] = 0;  v[-1, :] = 0;  v[:, 0] = 0;  v[:, -1] = 0
    # Moving lid (top wall)
    u[-1, :] = U_lid
    return u, v

print(f"Solver functions defined (all marked @pure for optimal caching)")

## 3. Build Sparse Operators

Constructing the Laplacian matrix is moderately expensive for large grids. Cash caches this so
re-running the cell (or any cell that depends on `N`, `dx`, `dy`) is instant if the grid hasn't changed.

**Multi-cell dependency:** This cell depends on `N`, `dx`, `dy` from Cell 4. If you change `N` above, Cash knows to recompute this.

In [ ]:
# Build the pressure Poisson operator (depends on N, dx, dy from Cell 4)
t0 = time.time()
A_laplacian = build_laplacian_2d(N, dx, dy)
print(f"Laplacian matrix: {A_laplacian.shape}, {A_laplacian.nnz} non-zeros")
print(f"Built in {time.time() - t0:.3f}s")

# Pre-factor the matrix for fast repeated solves
t0 = time.time()
from scipy.sparse.linalg import factorized
pressure_solve = factorized(A_laplacian)
print(f"LU factorization done in {time.time() - t0:.3f}s")

## 4. Time-Stepping Loop with Per-Iteration Caching

This is the core solver. Cash caches **each iteration of the loop independently**, so:
- If you change `n_steps` from 500 to 600, the first 500 iterations are instant (cached)
- If you interrupt mid-run, completed iterations are preserved
- Each iteration's cache key includes the previous state, so changing `Re` correctly invalidates everything

We solve using fractional-step (Chorin's projection) method:
1. **Advection-diffusion** step (explicit Euler)
2. **Pressure projection** (solve Poisson equation)
3. **Velocity correction** (enforce divergence-free)

In [ ]:
# Initialize velocity and pressure fields
u = np.zeros((N, N))   # x-velocity
v = np.zeros((N, N))   # y-velocity
p = np.zeros((N, N))   # pressure
u, v = apply_boundary_conditions(u, v, U_lid)
print(f"Initial fields: u_max={u.max():.2f}, v_max={v.max():.2f}")

# Time-stepping loop — each iteration is cached independently!
t_start = time.time()
residual_history = []

for step in range(n_steps):
    # 1. Compute advection terms (nonlinear convection)
    u_adv = u.copy()
    v_adv = v.copy()
    
    u_adv[1:-1, 1:-1] = (u[1:-1, 1:-1]
        - dt * u[1:-1, 1:-1] * (u[1:-1, 2:] - u[1:-1, :-2]) / (2 * dx)
        - dt * v[1:-1, 1:-1] * (u[2:, 1:-1] - u[:-2, 1:-1]) / (2 * dy)
        + dt * nu * (
            (u[1:-1, 2:] - 2*u[1:-1, 1:-1] + u[1:-1, :-2]) / dx**2 +
            (u[2:, 1:-1] - 2*u[1:-1, 1:-1] + u[:-2, 1:-1]) / dy**2
        ))
    
    v_adv[1:-1, 1:-1] = (v[1:-1, 1:-1]
        - dt * u[1:-1, 1:-1] * (v[1:-1, 2:] - v[1:-1, :-2]) / (2 * dx)
        - dt * v[1:-1, 1:-1] * (v[2:, 1:-1] - v[:-2, 1:-1]) / (2 * dy)
        + dt * nu * (
            (v[1:-1, 2:] - 2*v[1:-1, 1:-1] + v[1:-1, :-2]) / dx**2 +
            (v[2:, 1:-1] - 2*v[1:-1, 1:-1] + v[:-2, 1:-1]) / dy**2
        ))
    
    # 2. Pressure Poisson solve
    div = compute_divergence(u_adv, v_adv, dx, dy)
    rhs = (rho / dt) * div.flatten()
    p_flat = pressure_solve(rhs)
    p = p_flat.reshape((N, N))
    
    # 3. Velocity correction (projection)
    u[1:-1, 1:-1] = u_adv[1:-1, 1:-1] - (dt / rho) * (p[1:-1, 2:] - p[1:-1, :-2]) / (2 * dx)
    v[1:-1, 1:-1] = v_adv[1:-1, 1:-1] - (dt / rho) * (p[2:, 1:-1] - p[:-2, 1:-1]) / (2 * dy)
    
    # Apply boundary conditions
    u, v = apply_boundary_conditions(u, v, U_lid)
    u += 12
    
    # Track convergence
    residual = np.sqrt(np.mean(div[1:-1, 1:-1]**2))
    residual_history.append(residual)
    
    if step % 1000 == 0:
        print(f"  Step {step:4d}/{n_steps}: residual={residual:.2e}, "
              f"|u|_max={np.max(np.abs(u)):.4f}, |v|_max={np.max(np.abs(v)):.4f}")

wall_time = time.time() - t_start
print(f"\nSimulation complete in {wall_time:.2f}s")
print(f"Final residual: {residual_history[-1]:.2e}")

## 5. Post-Processing: Derived Quantities

Compute vorticity, stream function, and kinetic energy from the velocity field.
These depend on `u` and `v` — if the simulation parameters change upstream, Cash
automatically invalidates and recomputes these.

**Statement-level caching:** Each computation below is an independent statement.
If you modify only the stream function calculation, the vorticity result is still cached!

In [ ]:
# Statement 1: Vorticity field  ω = ∂v/∂x - ∂u/∂y
t0 = time.time()
vorticity = np.zeros((N, N))
vorticity[1:-1, 1:-1] = (
    (v[1:-1, 2:] - v[1:-1, :-2]) / (2 * dx) -
    (u[2:, 1:-1] - u[:-2, 1:-1]) / (2 * dy)
)
print(f"Vorticity: min={vorticity.min():.4f}, max={vorticity.max():.4f} (computed in {time.time()-t0:.3f}s)")

# Statement 2: Stream function via Poisson solve  ∇²ψ = -ω
t0 = time.time()
psi_flat = spsolve(A_laplacian, -vorticity.flatten())
stream_function = psi_flat.reshape((N, N))
print(f"Stream function: min={stream_function.min():.6f}, max={stream_function.max():.6f} (computed in {time.time()-t0:.3f}s)")

# Statement 3: Kinetic energy field  E = 0.5 * ρ * (u² + v²)
t0 = time.time()
kinetic_energy = 0.5 * rho * (u**2 + v**2)
total_KE = np.sum(kinetic_energy) * dx * dy
print(f"Total kinetic energy: {total_KE:.6f} J (computed in {time.time()-t0:.3f}s)")

## 6. Parameter Study with Conditional Branching

Cash caches **each branch of an if/else** separately. Here we perform different analyses
depending on whether the flow is laminar or transitional.

The `@cash:no-cache` annotation forces fresh computation of the diagnostic print (useful for
always showing current state), while the expensive analysis is cached.

> **💡 Try this:** Change `Re` in Cell 4 from 100 to 1000. The conditional branch switches from
> laminar to transitional analysis — only the new branch computes!

In [ ]:
# @cash:no-cache
print(f"\n{'='*50}")
print(f"Flow Regime Analysis for Re = {Re}")
print(f"{'='*50}")

if Re < 200:
    # Laminar regime — smooth streamlines, predictable vortex
    regime = "laminar"
    
    # Find primary vortex center (location of min stream function)
    iy, ix = np.unravel_index(np.argmin(stream_function), stream_function.shape)
    vortex_x, vortex_y = x[ix], y[iy]
    vortex_strength = vorticity[iy, ix]
    
    # Compute velocity profile along vertical centerline
    mid_x = N // 2
    u_centerline = u[:, mid_x]
    
    print(f"Regime: {regime}")
    print(f"Primary vortex center: ({vortex_x:.3f}, {vortex_y:.3f})")
    print(f"Vortex strength: {vortex_strength:.4f}")
    print(f"Centerline u range: [{u_centerline.min():.4f}, {u_centerline.max():.4f}]")

elif Re < 500:
    # Moderate Re — check for secondary corner vortices
    regime = "moderate"
    
    # Bottom-left corner vortex detection
    corner_region = stream_function[:N//4, :N//4]
    has_secondary = np.any(corner_region > 0) and np.any(corner_region < 0)
    
    # Enstrophy (integral of vorticity²) — measures flow complexity
    enstrophy = 0.5 * np.sum(vorticity**2) * dx * dy
    
    print(f"Regime: {regime}")
    print(f"Secondary corner vortex detected: {has_secondary}")
    print(f"Enstrophy: {enstrophy:.6f}")

else:
    # High Re — transitional, compute energy spectrum
    regime = "transitional"
    
    # 2D FFT of velocity magnitude for spectral analysis
    speed = np.sqrt(u**2 + v**2)
    speed_fft = np.fft.fft2(speed[1:-1, 1:-1])
    power_spectrum = np.abs(speed_fft)**2
    
    # Radially-averaged energy spectrum
    ny, nx_fft = power_spectrum.shape
    kx = np.fft.fftfreq(nx_fft, d=dx)
    ky = np.fft.fftfreq(ny, d=dy)
    KX, KY = np.meshgrid(kx, ky)
    K_mag = np.sqrt(KX**2 + KY**2)
    
    # Bin the spectrum
    k_bins = np.linspace(0, K_mag.max(), 20)
    spectrum_binned = np.zeros(len(k_bins) - 1)
    for i in range(len(k_bins) - 1):
        mask = (K_mag >= k_bins[i]) & (K_mag < k_bins[i+1])
        if mask.any():
            spectrum_binned[i] = np.mean(power_spectrum[mask])
    
    print(f"Regime: {regime}")
    print(f"Peak wavenumber: {k_bins[np.argmax(spectrum_binned)+1]:.1f}")
    print(f"Total spectral energy: {np.sum(spectrum_binned):.2e}")

print(f"\nKinetic energy: {total_KE:.6f} J")
print(f"Max velocity magnitude: {np.max(np.sqrt(u**2 + v**2)):.4f} m/s")

## 7. Grid Convergence Study with TTL

We run a quick grid convergence study at multiple resolutions. The `@cash:ttl=300` annotation
means these results are cached for 5 minutes — useful for convergence studies where you might
want to recompute periodically to verify stability.

The **loop iteration caching** means each grid resolution is cached independently:
- Add a new resolution? Only the new one computes.
- Remove one? The others stay cached.

In [ ]:
# @cash:ttl=300
# Grid convergence study — each resolution cached independently
resolutions = [11, 21, 31, 41]
convergence_results = {}

for n_grid in resolutions:
    t0 = time.time()
    dx_c = L / (n_grid - 1)
    dy_c = L / (n_grid - 1)
    
    # Quick 200-step simulation at this resolution
    u_c = np.zeros((n_grid, n_grid))
    v_c = np.zeros((n_grid, n_grid))
    u_c[-1, :] = U_lid  # lid BC
    
    A_c = build_laplacian_2d(n_grid, dx_c, dy_c)
    solve_c = factorized(A_c)
    
    for s in range(200):
        u_tmp = u_c.copy()
        v_tmp = v_c.copy()
        u_tmp[1:-1, 1:-1] = (u_c[1:-1, 1:-1]
            - dt * u_c[1:-1, 1:-1] * (u_c[1:-1, 2:] - u_c[1:-1, :-2]) / (2*dx_c)
            - dt * v_c[1:-1, 1:-1] * (u_c[2:, 1:-1] - u_c[:-2, 1:-1]) / (2*dy_c)
            + dt * nu * ((u_c[1:-1, 2:] - 2*u_c[1:-1, 1:-1] + u_c[1:-1, :-2])/dx_c**2
                       + (u_c[2:, 1:-1] - 2*u_c[1:-1, 1:-1] + u_c[:-2, 1:-1])/dy_c**2))
        v_tmp[1:-1, 1:-1] = (v_c[1:-1, 1:-1]
            - dt * u_c[1:-1, 1:-1] * (v_c[1:-1, 2:] - v_c[1:-1, :-2]) / (2*dx_c)
            - dt * v_c[1:-1, 1:-1] * (v_c[2:, 1:-1] - v_c[:-2, 1:-1]) / (2*dy_c)
            + dt * nu * ((v_c[1:-1, 2:] - 2*v_c[1:-1, 1:-1] + v_c[1:-1, :-2])/dx_c**2
                       + (v_c[2:, 1:-1] - 2*v_c[1:-1, 1:-1] + v_c[:-2, 1:-1])/dy_c**2))
        
        div_c = compute_divergence(u_tmp, v_tmp, dx_c, dy_c)
        p_c = solve_c((rho/dt) * div_c.flatten()).reshape((n_grid, n_grid))
        
        u_c[1:-1,1:-1] = u_tmp[1:-1,1:-1] - (dt/rho)*(p_c[1:-1,2:]-p_c[1:-1,:-2])/(2*dx_c)
        v_c[1:-1,1:-1] = v_tmp[1:-1,1:-1] - (dt/rho)*(p_c[2:,1:-1]-p_c[:-2,1:-1])/(2*dy_c)
        u_c, v_c = apply_boundary_conditions(u_c, v_c, U_lid)
    
    # Measure: centerline u-velocity at mid-height
    mid = n_grid // 2
    u_mid = u_c[mid, mid]
    
    elapsed = time.time() - t0
    convergence_results[n_grid] = {
        'u_center': u_mid,
        'max_speed': np.max(np.sqrt(u_c**2 + v_c**2)),
        'time': elapsed
    }
    print(f"  N={n_grid:3d}: u_center={u_mid:.6f}, max_speed={convergence_results[n_grid]['max_speed']:.4f}, time={elapsed:.2f}s")

# Richardson extrapolation between two finest grids
if len(resolutions) >= 2:
    u_fine = convergence_results[resolutions[-1]]['u_center']
    u_coarse = convergence_results[resolutions[-2]]['u_center']
    r = resolutions[-1] / resolutions[-2]  # refinement ratio
    order = np.log(abs((convergence_results[resolutions[-3]]['u_center'] - u_coarse) / 
                       (u_coarse - u_fine + 1e-15))) / np.log(r) if len(resolutions) >= 3 else 2.0
    u_exact_est = u_fine + (u_fine - u_coarse) / (r**order - 1)
    print(f"\nRichardson extrapolation: u_exact ≈ {u_exact_est:.6f} (order ≈ {order:.1f})")

## 8. Stochastic Initial Conditions

For turbulence studies, we often add random perturbations to initial conditions.
The `@cash:allow-random` annotation tells Cash to cache despite the random calls
(which it would otherwise flag as non-deterministic).

We seed the RNG for reproducibility, but `@cash:allow-random` is still needed because
Cash's randomness detector flags *any* random call by default.

In [ ]:
# @cash:allow-random
# Perturbed initial condition for studying sensitivity
rng = np.random.default_rng(seed=42)
perturbation_amplitude = 0.01

u_perturbed = np.zeros((N, N)) + perturbation_amplitude * rng.standard_normal((N, N))
v_perturbed = np.zeros((N, N)) + perturbation_amplitude * rng.standard_normal((N, N))
u_perturbed, v_perturbed = apply_boundary_conditions(u_perturbed, v_perturbed, U_lid)

# Run 200 steps with perturbed IC
t0 = time.time()
for s in range(200):
    u_tmp = u_perturbed.copy()
    v_tmp = v_perturbed.copy()
    u_tmp[1:-1,1:-1] = (u_perturbed[1:-1,1:-1]
        - dt*u_perturbed[1:-1,1:-1]*(u_perturbed[1:-1,2:]-u_perturbed[1:-1,:-2])/(2*dx)
        - dt*v_perturbed[1:-1,1:-1]*(u_perturbed[2:,1:-1]-u_perturbed[:-2,1:-1])/(2*dy)
        + dt*nu*((u_perturbed[1:-1,2:]-2*u_perturbed[1:-1,1:-1]+u_perturbed[1:-1,:-2])/dx**2
                +(u_perturbed[2:,1:-1]-2*u_perturbed[1:-1,1:-1]+u_perturbed[:-2,1:-1])/dy**2))
    v_tmp[1:-1,1:-1] = (v_perturbed[1:-1,1:-1]
        - dt*u_perturbed[1:-1,1:-1]*(v_perturbed[1:-1,2:]-v_perturbed[1:-1,:-2])/(2*dx)
        - dt*v_perturbed[1:-1,1:-1]*(v_perturbed[2:,1:-1]-v_perturbed[:-2,1:-1])/(2*dy)
        + dt*nu*((v_perturbed[1:-1,2:]-2*v_perturbed[1:-1,1:-1]+v_perturbed[1:-1,:-2])/dx**2
                +(v_perturbed[2:,1:-1]-2*v_perturbed[1:-1,1:-1]+v_perturbed[:-2,1:-1])/dy**2))
    
    div_p = compute_divergence(u_tmp, v_tmp, dx, dy)
    p_p = pressure_solve((rho/dt)*div_p.flatten()).reshape((N, N))
    u_perturbed[1:-1,1:-1] = u_tmp[1:-1,1:-1] - (dt/rho)*(p_p[1:-1,2:]-p_p[1:-1,:-2])/(2*dx)
    v_perturbed[1:-1,1:-1] = v_tmp[1:-1,1:-1] - (dt/rho)*(p_p[2:,1:-1]-p_p[:-2,1:-1])/(2*dy)
    u_perturbed, v_perturbed = apply_boundary_conditions(u_perturbed, v_perturbed, U_lid)

perturbed_time = time.time() - t0
print(f"Perturbed simulation: {perturbed_time:.2f}s")

# Compare with unperturbed solution
u_diff = np.max(np.abs(u_perturbed - u))
v_diff = np.max(np.abs(v_perturbed - v))
print(f"Max velocity difference from unperturbed:")
print(f"  |Δu|_max = {u_diff:.6e}")
print(f"  |Δv|_max = {v_diff:.6e}")
print(f"  Perturbation amplification factor: {max(u_diff, v_diff) / perturbation_amplitude:.2f}x")

## 9. Save & Reload Results (File Dependency Tracking)

Cash tracks file dependencies through `np.save` and `np.load`. If the saved file changes
on disk, the loading cell automatically recomputes.

This is critical for CFD workflows where:
- You save intermediate results to disk
- A colleague modifies the data file
- Cash detects the change and invalidates the cache

In [ ]:
# Save simulation results to disk
results_dir = os.path.join(os.path.dirname(os.path.abspath('.')), 'examples', '.cash_cfd_results')
os.makedirs(results_dir, exist_ok=True)

result_file = os.path.join(results_dir, f'cavity_Re{Re}_N{N}.npz')
np.savez(result_file,
         u=u, v=v, p=p, vorticity=vorticity,
         stream_function=stream_function,
         x=x, y=y, Re=Re, N=N)
file_size = os.path.getsize(result_file)
print(f"Saved results to: {result_file}")
print(f"File size: {file_size / 1024:.1f} KB")

In [ ]:
# Reload and validate — Cash tracks this file dependency!
# If the .npz file changes on disk, this cell auto-recomputes.
loaded = np.load(result_file)
u_loaded = loaded['u']
v_loaded = loaded['v']

# Verify integrity
u_match = np.allclose(u, u_loaded)
v_match = np.allclose(v, v_loaded)
print(f"Loaded Re={int(loaded['Re'])}, N={int(loaded['N'])}")
print(f"Integrity check: u={'✓' if u_match else '✗'}, v={'✓' if v_match else '✗'}")

# Compute derived quantity from loaded data
speed_loaded = np.sqrt(u_loaded**2 + v_loaded**2)
print(f"Max speed from loaded data: {speed_loaded.max():.4f} m/s")

## 10. Multi-Resolution Comparison (Nested Loops)

Nested loop caching: each `(metric, resolution)` combination gets its own cache entry.
Add a new metric? Only iterations for that metric compute — existing ones are cached.

In [ ]:
# Nested loop — each combination cached independently
metrics_to_compute = ['max_vorticity', 'kinetic_energy', 'enstrophy']
test_resolutions = [21, 31, 41]

comparison_table = {}
for metric_name in metrics_to_compute:
    comparison_table[metric_name] = {}
    for n_res in test_resolutions:
        # Use stored convergence results where available
        if n_res == N:  # Use the main simulation
            u_r, v_r = u, v
        elif n_res in convergence_results:
            # These were computed in the convergence study
            u_r = np.zeros((n_res, n_res))  # Placeholder — in practice, store full fields
            v_r = np.zeros((n_res, n_res))
        else:
            u_r = np.zeros((n_res, n_res))
            v_r = np.zeros((n_res, n_res))
        
        dx_r = L / (n_res - 1)
        dy_r = L / (n_res - 1)
        
        if metric_name == 'max_vorticity':
            w = np.zeros((n_res, n_res))
            w[1:-1, 1:-1] = ((v_r[1:-1,2:]-v_r[1:-1,:-2])/(2*dx_r) - 
                             (u_r[2:,1:-1]-u_r[:-2,1:-1])/(2*dy_r))
            value = np.max(np.abs(w))
        elif metric_name == 'kinetic_energy':
            value = 0.5 * np.sum(u_r**2 + v_r**2) * dx_r * dy_r
        elif metric_name == 'enstrophy':
            w = np.zeros((n_res, n_res))
            w[1:-1, 1:-1] = ((v_r[1:-1,2:]-v_r[1:-1,:-2])/(2*dx_r) - 
                             (u_r[2:,1:-1]-u_r[:-2,1:-1])/(2*dy_r))
            value = 0.5 * np.sum(w**2) * dx_r * dy_r
        else:
            value = 0.0
        
        comparison_table[metric_name][n_res] = value
        print(f"  {metric_name:>20s} @ N={n_res:3d}: {value:.6e}")

print("\n=== Summary Table ===")
header = f"{'Metric':>20s}" + "".join(f"{'N='+str(n):>14s}" for n in test_resolutions)
print(header)
print("-" * len(header))
for metric_name in metrics_to_compute:
    row = f"{metric_name:>20s}"
    for n_res in test_resolutions:
        row += f"{comparison_table[metric_name][n_res]:14.6e}"
    print(row)

## 11. Convergence History Analysis

Analyze the time-stepping convergence. This cell depends on `residual_history` from
Cell 8 — changing simulation parameters propagates here automatically.

In [ ]:
# Convergence diagnostics
residuals = np.array(residual_history)
print(f"Convergence History (Re={Re}, N={N}, {n_steps} steps):")
print(f"  Initial residual:  {residuals[0]:.6e}")
print(f"  Final residual:    {residuals[-1]:.6e}")
print(f"  Reduction factor:  {residuals[0] / (residuals[-1] + 1e-15):.1f}x")
print(f"  Min residual:      {residuals.min():.6e} (at step {residuals.argmin()})")

# Compute convergence rate (exponential fit to last 50% of history)
half = len(residuals) // 2
if residuals[half:].min() > 0:
    log_res = np.log(residuals[half:])
    steps_arr = np.arange(half, len(residuals))
    coeffs = np.polyfit(steps_arr, log_res, 1)
    convergence_rate = coeffs[0]
    print(f"  Convergence rate:  {convergence_rate:.6f} (exponential decay constant)")
    print(f"  Half-life:         {-np.log(2)/convergence_rate:.0f} steps")
else:
    print(f"  Residual reached machine zero — fully converged!")

# Velocity field statistics
speed = np.sqrt(u**2 + v**2)

print(f"\nVelocity Field Statistics:")
print(f"  Mean speed:     {speed.mean():.6f} m/s")
print(f"  Max speed:      {speed.max():.6f} m/s")
print(f"  RMS velocity:   {np.sqrt(np.mean(u**2 + v**2)):.6f} m/s")
print(f"  Max |u|:        {np.max(np.abs(u)):.6f} m/s")
print(f"  Max |v|:        {np.max(np.abs(v)):.6f} m/s")

## 12. Session Summary

Use `%cash_stats` to see how much time caching saved across the entire session.
On a second run, you should see significant time savings from cached results!

In [ ]:
%cash_stats

---

## Summary of Caching Interactions

This notebook showcases how Cash's caching features work together in a real scientific computing workflow:

### Dependency Chain
```
Cell 4 (Re, N, grid)  →  Cell 6 (solver functions)  →  Cell 7 (Laplacian)
                       →  Cell 8 (simulation loop)   →  Cell 9 (post-processing)
                       →  Cell 10 (regime analysis)  →  Cell 11 (convergence study)
                       →  Cell 14 (perturbed IC)     →  Cell 17 (diagnostics)
```

### What Happens When You Change Parameters
| Change | Effect |
|--------|--------|
| Change `Re` in Cell 4 | Everything downstream recomputes (physics changed) |
| Change `N` in Cell 4 | Grid + operators + simulation recompute |
| Change `n_steps` in Cell 4 | Simulation recomputes, but Laplacian is cached |
| Add resolution to convergence study | Only new resolution computes |
| Add metric to nested loop | Only new metric computes at all resolutions |
| Modify post-processing only | Simulation is cached, only post-processing reruns |
| Restart kernel | `@cash:persist` cells restore from disk |
| External file changes | `np.load` cells auto-invalidate |